# Lab 8 - Parameterized Gold Star Schema

This notebook is the Lab 8 parameterized copy of the existing Lab 6 Gold processing. It preserves the six Lab 6 Gold transformations while taking the catalog and Bronze, Silver, and Gold schemas as notebook parameters.

The dashboard, alert, governance, and validation cells from Lab 6 are intentionally not included here.

In [ ]:
dbutils.widgets.text("catalog", "lab5", "Catalog")
dbutils.widgets.text("bronze_schema", "bronze", "Bronze schema")
dbutils.widgets.text("silver_schema", "silver", "Silver schema")
dbutils.widgets.text("gold_schema", "gold", "Gold schema")

catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
silver_schema = dbutils.widgets.get("silver_schema")
gold_schema = dbutils.widgets.get("gold_schema")

bronze_namespace = f"{catalog}.{bronze_schema}"
silver_namespace = f"{catalog}.{silver_schema}"
gold_namespace = f"{catalog}.{gold_schema}"

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {gold_namespace}.dim_date
COMMENT 'Date dimension table covering sales order period (July-Dec 2019) with standard calendar attributes'
AS
WITH date_spine AS (
  SELECT
    EXPLODE(
      SEQUENCE(
        TO_DATE('2019-07-01'),
        TO_DATE('2019-12-31'),
        INTERVAL 1 DAY
      )
    ) AS full_date
)
SELECT
  CAST(DATE_FORMAT(full_date, 'yyyyMMdd') AS INT) AS date_key,
  full_date,
  YEAR(full_date) AS year,
  QUARTER(full_date) AS quarter,
  MONTH(full_date) AS month,
  DATE_FORMAT(full_date, 'MMMM') AS month_name,
  WEEKOFYEAR(full_date) AS week_of_year,
  DAYOFMONTH(full_date) AS day_of_month,
  DAYOFWEEK(full_date) AS day_of_week,
  DATE_FORMAT(full_date, 'EEEE') AS day_name,
  CASE WHEN DAYOFWEEK(full_date) IN (1, 7) THEN TRUE ELSE FALSE END AS is_weekend
FROM date_spine
ORDER BY full_date
""")

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {gold_namespace}.dim_customers
COMMENT 'Current customer dimension with demographics, loyalty status, and order flags for Lab 6 Gold analytics'
AS
WITH current_customers AS (
  SELECT
    customer_id, customer_name, tax_id, tax_code, state, city, region, postcode,
    loyalty_segment, units_purchased, valid_from_ts,
    ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY valid_from_ts DESC) AS rn
  FROM {silver_namespace}.slv_customers_clean
  WHERE valid_to_ts IS NULL
),
customers_with_orders AS (
  SELECT DISTINCT customer_id
  FROM {silver_namespace}.slv_sales_orders_clean
)
SELECT
  ABS(HASH(c.customer_id)) AS customer_key,
  c.customer_id, c.customer_name, c.tax_id, c.tax_code, c.state, c.city,
  c.region, c.postcode, c.loyalty_segment,
  CASE c.loyalty_segment
    WHEN 0 THEN 'None'
    WHEN 1 THEN 'Bronze'
    WHEN 2 THEN 'Silver'
    WHEN 3 THEN 'Gold'
    ELSE 'Unknown'
  END AS loyalty_segment_name,
  c.units_purchased,
  CASE WHEN o.customer_id IS NOT NULL THEN TRUE ELSE FALSE END AS has_orders,
  DATE(c.valid_from_ts) AS effective_date,
  TRUE AS is_current
FROM current_customers c
LEFT JOIN customers_with_orders o ON c.customer_id = o.customer_id
WHERE c.rn = 1
ORDER BY c.customer_id
""")

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {gold_namespace}.dim_products
COMMENT 'Product dimension with ordering history for Lab 6 Gold analytics'
AS
WITH exploded_products AS (
  SELECT DATE(order_ts) AS order_date, product.id AS product_id, product.name AS product_name
  FROM {silver_namespace}.slv_sales_orders_clean
  LATERAL VIEW EXPLODE(ordered_products) AS product
),
product_aggregates AS (
  SELECT product_id, product_name, MIN(order_date) AS first_order_date,
    MAX(order_date) AS last_order_date, COUNT(*) AS times_ordered
  FROM exploded_products
  GROUP BY product_id, product_name
)
SELECT ABS(HASH(product_id)) AS product_key, product_id, product_name,
  first_order_date, last_order_date, times_ordered
FROM product_aggregates
ORDER BY product_id
""")

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {gold_namespace}.fct_sales_orders
COMMENT 'Sales order line items fact table with one row per product ordered'
AS
WITH exploded_orders AS (
  SELECT o.order_number, o.customer_id, o.order_ts, product.id AS product_id,
    product.name AS product_name, product.price AS price_cents, product.qty AS quantity,
    product.curr AS currency, product.unit AS unit_of_measure,
    product.promotion_info.promo_disc AS promo_disc_pct,
    product.promotion_info.promo_id AS promo_id,
    ROW_NUMBER() OVER (PARTITION BY o.order_number ORDER BY product.id) AS line_sequence
  FROM {silver_namespace}.slv_sales_orders_clean o
  LATERAL VIEW EXPLODE(o.ordered_products) AS product
)
SELECT e.order_number * 1000 + e.line_sequence AS order_line_key,
  e.order_number, c.customer_key, p.product_key, d.date_key, e.order_ts, e.product_id,
  e.quantity,
  CAST(e.price_cents / 100.0 AS DECIMAL(10,2)) AS unit_price,
  CAST((e.price_cents * e.quantity) / 100.0 AS DECIMAL(10,2)) AS line_total,
  CAST(COALESCE(e.price_cents * e.quantity * e.promo_disc_pct, 0) / 100.0 AS DECIMAL(10,2)) AS promotion_discount,
  COALESCE(e.promo_id, 0) AS promotion_id,
  CASE WHEN e.promo_disc_pct IS NOT NULL AND e.promo_disc_pct > 0 THEN TRUE ELSE FALSE END AS has_promotion,
  e.currency, e.unit_of_measure
FROM exploded_orders e
INNER JOIN {gold_namespace}.dim_customers c ON e.customer_id = c.customer_id
INNER JOIN {gold_namespace}.dim_products p ON e.product_id = p.product_id
INNER JOIN {gold_namespace}.dim_date d ON DATE(e.order_ts) = d.full_date
ORDER BY e.order_number, e.line_sequence
""")

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {gold_namespace}.agg_customer_summary
COMMENT 'Customer-level sales summary with lifetime value metrics for customers who have placed orders'
AS
WITH customer_metrics AS (
  SELECT f.customer_key, COUNT(DISTINCT f.order_number) AS total_orders,
    COUNT(*) AS total_line_items, SUM(f.quantity) AS total_quantity,
    SUM(f.line_total) AS total_revenue, MIN(DATE(f.order_ts)) AS first_order_date,
    MAX(DATE(f.order_ts)) AS last_order_date
  FROM {gold_namespace}.fct_sales_orders f
  GROUP BY f.customer_key
)
SELECT cm.customer_key, c.customer_id, c.customer_name, c.state, c.loyalty_segment_name,
  cm.total_orders, cm.total_line_items, cm.total_quantity,
  CAST(cm.total_revenue AS DECIMAL(10,2)) AS total_revenue,
  CAST(cm.total_revenue / cm.total_orders AS DECIMAL(10,2)) AS avg_order_value,
  cm.first_order_date, cm.last_order_date,
  DATEDIFF(CURRENT_DATE(), cm.last_order_date) AS days_since_last_order
FROM customer_metrics cm
INNER JOIN {gold_namespace}.dim_customers c ON cm.customer_key = c.customer_key
ORDER BY cm.total_revenue DESC
""")

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {gold_namespace}.agg_daily_sales
COMMENT 'Daily sales summary with order volume, revenue, and customer metrics for dashboard and alerting'
AS
WITH daily_metrics AS (
  SELECT DATE(f.order_ts) AS order_date, COUNT(DISTINCT f.order_number) AS total_orders,
    COUNT(*) AS total_line_items, SUM(f.quantity) AS total_quantity,
    SUM(f.line_total) AS total_revenue, COUNT(DISTINCT f.customer_key) AS unique_customers
  FROM {gold_namespace}.fct_sales_orders f
  GROUP BY DATE(f.order_ts)
)
SELECT dm.order_date, d.date_key, dm.total_orders, dm.total_line_items, dm.total_quantity,
  dm.unique_customers,
  CAST(dm.total_revenue AS DECIMAL(10,2)) AS total_revenue,
  CAST(dm.total_revenue / dm.total_orders AS DECIMAL(10,2)) AS avg_order_value
FROM daily_metrics dm
INNER JOIN {gold_namespace}.dim_date d ON dm.order_date = d.full_date
ORDER BY dm.order_date
""")